<a href="https://colab.research.google.com/github/ramzanr12/Age-prediction/blob/Ramzan/MiVOLO_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install dependencies (matching MiVOLO's exact versions)
!pip install -q timm==0.8.13.dev0 huggingface_hub
!git clone -q https://github.com/WildChlamydia/MiVOLO.git /content/MiVOLO
%cd /content/MiVOLO
!pip install -q -e . --no-deps
import sys
sys.path.insert(0, "/content/MiVOLO")
print("Setup done.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.1 MB/s eta 0:00:00
/content/MiVOLO
  Preparing metadata (setup.py) ... done
Setup done.


In [2]:
# Cell 2: Imports
import os, re, random, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

from mivolo.model.create_timm_model import create_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [4]:
# Cell 3: Set up your Kaggle API token (new token-based auth)
# Get a fresh token from kaggle.com -> Settings -> API -> Create New Token
import os

KAGGLE_API_TOKEN = "KGAT_304a859e4b56c1482859bed919008322"  # looks like: KGAT_xxxxxxxxxxxxxxxxxxxxx

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(KAGGLE_API_TOKEN)
!chmod 600 /root/.kaggle/access_token

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
print("Kaggle token configured.")


Kaggle token configured.


In [5]:
# Cell 4: Download the Kaggle dataset
!pip install -q kaggle
!kaggle datasets download -d mariafrenti/age-prediction -p /content/data --unzip
print("Dataset downloaded to /content/data")

# Quick look at what got downloaded
for root, dirs, filenames in os.walk("/content/data"):
    depth = root.replace("/content/data", "").count(os.sep)
    if depth <= 2:
        print(root, "->", len(filenames), "files,", len(dirs), "subfolders")


Dataset URL: https://www.kaggle.com/datasets/mariafrenti/age-prediction
License(s): unknown
100% 2.03G/2.03G [00:51<00:00, 41.9MB/s]

Dataset downloaded to /content/data
/content/data -> 0 files, 2 subfolders
/content/data/20-50 -> 0 files, 1 subfolders
/content/data/20-50/20-50 -> 0 files, 2 subfolders
/content/data/age_prediction_up -> 0 files, 1 subfolders
/content/data/age_prediction_up/age_prediction -> 0 files, 2 subfolders


In [6]:
# Cell 5: Download the face-only, age-only pretrained checkpoint (from MiVOLO README)
# This is the correct starting point since your dataset is face-only images with only age labels.
!pip install -q gdown
!gdown --fuzzy "https://drive.google.com/file/d/17ysOqgG3FUyEuxrV3Uh49EpmuOiGDxrq/view?usp=drive_link" -O /content/mivolo_face_age.pth.tar

ckpt_path = "/content/mivolo_face_age.pth.tar"
assert os.path.exists(ckpt_path), "Download failed - if this cell errors, manually download from the MiVOLO README and upload it via files.upload()"
print("Checkpoint ready:", ckpt_path)


Downloading...
From: https://drive.google.com/uc?id=17ysOqgG3FUyEuxrV3Uh49EpmuOiGDxrq
To: /content/mivolo_face_age.pth.tar
100% 103M/103M [00:00<00:00, 184MB/s] 
Checkpoint ready: /content/mivolo_face_age.pth.tar


In [7]:
# Cell 6: Inspect checkpoint metadata (auto-detects everything, no hardcoding needed)
state = torch.load(ckpt_path, map_location="cpu")
min_age = state["min_age"]
max_age = state["max_age"]
avg_age = state["avg_age"]
no_gender = state["no_gender"]
with_persons_model = state.get("with_persons_model", "patch_embed.conv1.0.weight" in state["state_dict"])
input_size = state["state_dict"]["pos_embed"].shape[1] * 16

num_classes = 1 if no_gender else 3
in_chans = 6 if with_persons_model else 3

print(f"min_age={min_age}, max_age={max_age}, avg_age={avg_age}")
print(f"only_age={no_gender}, with_persons_model={with_persons_model}")
print(f"input_size={input_size}, num_classes={num_classes}, in_chans={in_chans}")

assert in_chans == 3, "This checkpoint is not face-only. Re-check Cell 5's download."


min_age=1, max_age=95, avg_age=48.0
only_age=True, with_persons_model=False
input_size=224, num_classes=1, in_chans=3


In [9]:
# Cell 7: Collect samples, filter corrupted images, check age distribution
DATA_ROOT = "/content/data"

def collect_samples(root_dir):
    samples = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        folder_name = os.path.basename(dirpath)
        if re.fullmatch(r"\d+", folder_name):
            age = int(folder_name)
            if not (min_age <= age <= max_age):
                continue
            for f in filenames:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    samples.append((os.path.join(dirpath, f), age))
    return samples

raw_samples = collect_samples(DATA_ROOT)
print("Raw samples found:", len(raw_samples))
assert len(raw_samples) > 0, "No age-labeled folders found. Check the dataset folder structure from Cell 4 output."

# Filter out corrupted/unreadable images
good_samples = []
bad_count = 0
for path, age in raw_samples:
    try:
        img = Image.open(path)
        img.verify()
        good_samples.append((path, age))
    except Exception:
        bad_count += 1

print(f"Corrupted/unreadable images removed: {bad_count}")
print(f"Usable samples: {len(good_samples)}")

# Age distribution check (helps explain accuracy later)
from collections import Counter
age_counts = Counter([a for _, a in good_samples])
print()
sparse_ages = [age for age in sorted(age_counts) if age_counts[age] < 5]
print("Age groups with very few images (<5):")
print(f"  {len(sparse_ages)} sparse age groups:", sparse_ages[:20])
print(f"Most common age: {age_counts.most_common(1)}")
print(f"Least common age: {min(age_counts.items(), key=lambda x: x[1])}")

all_samples = good_samples

# Split train/val - use existing test/ folder if present, else random 90/10 split
train_samples, val_samples = [], []
for path, age in all_samples:
    norm_path = path.replace("\\", "/").lower()
    if "/test/" in norm_path or "/val/" in norm_path:
        val_samples.append((path, age))
    else:
        train_samples.append((path, age))

if len(val_samples) == 0:
    random.seed(42)
    random.shuffle(train_samples)
    split = int(0.9 * len(train_samples))
    val_samples = train_samples[split:]
    train_samples = train_samples[:split]

print(f"\nTrain: {len(train_samples)}  Val: {len(val_samples)}")

Raw samples found: 273569
Corrupted/unreadable images removed: 0
Usable samples: 273569

Age groups with very few images (<5):
  0 sparse age groups: []
Most common age: [(31, 9846)]
Least common age: (95, 17)

Train: 219012  Val: 54557


In [13]:
!pip install -q ultralytics==8.1.0 lapx>=0.5.2 yt_dlp

In [15]:
import torch
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load

In [16]:
# New cell: Download YOLO face/person detector + crop faces from raw images
from huggingface_hub import hf_hub_download
import cv2
import numpy as np
from mivolo.model.yolo_detector import Detector

detector_path = hf_hub_download(repo_id="iitolstykh/demo_yolov8_detector", filename="yolov8x_person_face.pt")
detector = Detector(detector_path, device=device, verbose=False)
print("Face detector loaded.")

Model summary (fused): 268 layers, 68125494 parameters, 0 gradients, 257.4 GFLOPs
Face detector loaded.


In [17]:
# New cell: Run face detection on every image, save cropped faces to disk
CROP_DIR = "/content/data_cropped"
os.makedirs(CROP_DIR, exist_ok=True)

MARGIN = 0.15  # extra margin around detected face box (matches MiVOLO's typical crop style)
cropped_samples = []
no_face_count = 0

for i, (path, age) in enumerate(good_samples):
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        no_face_count += 1
        continue

    result = detector.predict(img_bgr)
    face_indices = [ind for ind in range(len(result.yolo_results.boxes))
                     if result.yolo_results.names[int(result.yolo_results.boxes[ind].cls)] == "face"]

    if not face_indices:
        # no face detected - fall back to the original full image
        crop = img_bgr
    else:
        # pick the largest detected face (most likely the main subject)
        best_ind = max(face_indices, key=lambda ind: result.get_bbox_by_ind(ind, *img_bgr.shape[:2])[2:].prod())
        x1, y1, x2, y2 = result.get_bbox_by_ind(best_ind, *img_bgr.shape[:2])
        h, w = img_bgr.shape[:2]
        bw, bh = x2 - x1, y2 - y1
        x1 = max(0, int(x1 - bw * MARGIN))
        y1 = max(0, int(y1 - bh * MARGIN))
        x2 = min(w, int(x2 + bw * MARGIN))
        y2 = min(h, int(y2 + bh * MARGIN))
        crop = img_bgr[y1:y2, x1:x2]

    out_path = os.path.join(CROP_DIR, f"{i}_{age}.jpg")
    cv2.imwrite(out_path, crop)
    cropped_samples.append((out_path, age))

    if i % 500 == 0:
        print(f"  processed {i}/{len(good_samples)} images  (no-face fallback count so far: {no_face_count})")

print(f"Done. {len(cropped_samples)} cropped images saved. {no_face_count} unreadable images skipped.")

# Replace good_samples with the face-cropped versions for training
good_samples = cropped_samples
all_samples = good_samples

# Re-split train/val since paths changed
train_samples, val_samples = [], []
random.seed(42)
random.shuffle(all_samples)
split = int(0.9 * len(all_samples))
train_samples = all_samples[:split]
val_samples = all_samples[split:]
print(f"Train: {len(train_samples)}  Val: {len(val_samples)}")

  processed 0/273569 images  (no-face fallback count so far: 0)
  processed 500/273569 images  (no-face fallback count so far: 0)
  processed 1000/273569 images  (no-face fallback count so far: 0)
  processed 1500/273569 images  (no-face fallback count so far: 0)
  processed 2000/273569 images  (no-face fallback count so far: 0)
  processed 2500/273569 images  (no-face fallback count so far: 0)
  processed 3000/273569 images  (no-face fallback count so far: 0)
  processed 3500/273569 images  (no-face fallback count so far: 0)
  processed 4000/273569 images  (no-face fallback count so far: 0)
  processed 4500/273569 images  (no-face fallback count so far: 0)
  processed 5000/273569 images  (no-face fallback count so far: 0)
  processed 5500/273569 images  (no-face fallback count so far: 0)
  processed 6000/273569 images  (no-face fallback count so far: 0)
  processed 6500/273569 images  (no-face fallback count so far: 0)
  processed 7000/273569 images  (no-face fallback count so far: 0)

In [18]:
# Cell 7b: Weighted sampler to balance rare age groups during training
from torch.utils.data import WeightedRandomSampler

train_ages = [age for _, age in train_samples]
age_counts_train = Counter(train_ages)

# inverse-frequency weight - rare ages get sampled more often, common ages less
sample_weights = [1.0 / age_counts_train[age] for age in train_ages]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
print("Weighted sampler ready - rare age groups will be sampled more often during training.")


Weighted sampler ready - rare age groups will be sampled more often during training.


In [19]:
# Cell 8: Dataset class + dataloaders (using weighted sampler for train)
class AgeDataset(Dataset):
    def __init__(self, samples, input_size, mean, std, train=True):
        self.samples = samples
        if train:
            self.tf = T.Compose([
                T.Resize((input_size, input_size)),
                T.RandomHorizontalFlip(),
                T.ColorJitter(brightness=0.15, contrast=0.15),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])
        else:
            self.tf = T.Compose([
                T.Resize((input_size, input_size)),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, age = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.tf(img)
        target = (age - avg_age) / (max_age - min_age)
        return img, torch.tensor(target, dtype=torch.float32), torch.tensor(age, dtype=torch.float32)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_ds = AgeDataset(train_samples, input_size, MEAN, STD, train=True)
val_ds = AgeDataset(val_samples, input_size, MEAN, STD, train=False)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("Dataloaders ready.")


Dataloaders ready.


In [22]:
# Cell 9: Build the model, load pretrained checkpoint, unfreeze head + final norm only
model = create_model(
    model_name=f"mivolo_d1_{input_size}",
    num_classes=num_classes,
    in_chans=in_chans,
    pretrained=False,
    checkpoint_path=ckpt_path,
    filter_keys=["fds."],
)
model = model.to(device)

# Freeze the backbone, only fine-tune the head + final norm layer.
for p in model.parameters():
    p.requires_grad = False
for name, p in model.named_parameters():
    if "head" in name or name.startswith("norm"):
        p.requires_grad = True

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("Trainable layers:", trainable)

Trainable layers: ['aux_head.weight', 'aux_head.bias', 'norm.weight', 'norm.bias', 'head.weight', 'head.bias']


In [23]:
# Cell 10: Fine-tune the model (MAE + Accuracy printed per epoch)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
criterion = nn.L1Loss()

NUM_EPOCHS = 8
best_val_mae = float("inf")
CKPT_OUT_DIR = "/content/checkpoints"
os.makedirs(CKPT_OUT_DIR, exist_ok=True)

total_batches = len(train_loader)
print(f"Starting training: {total_batches} batches per epoch, {NUM_EPOCHS} epochs total")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    t0 = time.time()
    for batch_idx, (imgs, targets, ages) in enumerate(train_loader):
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        if out.dim() > 1:
            out = out.squeeze(-1)
        loss = criterion(out, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

        if batch_idx % 20 == 0:
            elapsed = time.time() - t0
            print(f"  epoch {epoch} batch {batch_idx}/{total_batches}  loss={loss.item():.4f}  elapsed={elapsed:.0f}s")

    train_loss = running_loss / len(train_ds)

    model.eval()
    mae_sum = 0.0
    correct_within_5 = 0
    correct_exact = 0
    total = 0
    with torch.no_grad():
        for imgs, targets, ages in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            if out.dim() > 1:
                out = out.squeeze(-1)
            pred_age = out.cpu() * (max_age - min_age) + avg_age
            diff = torch.abs(pred_age - ages)
            mae_sum += diff.sum().item()
            correct_within_5 += (diff <= 5).sum().item()
            correct_exact += (diff.round() == 0).sum().item()
            total += ages.size(0)

    val_mae = mae_sum / total
    acc_cs5 = 100.0 * correct_within_5 / total
    acc_exact = 100.0 * correct_exact / total

    print(f"Epoch {epoch}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_age_MAE={val_mae:.2f} yrs  Accuracy(\u00b15yrs)={acc_cs5:.1f}%  "
          f"Accuracy(exact)={acc_exact:.1f}%  ({time.time()-t0:.0f}s)")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save({
            "state_dict": model.state_dict(),
            "min_age": min_age,
            "max_age": max_age,
            "avg_age": avg_age,
            "no_gender": no_gender,
            "with_persons_model": with_persons_model,
        }, f"{CKPT_OUT_DIR}/mivolo_finetuned_best.pth.tar")
        print(f"  -> saved new best checkpoint (val MAE {best_val_mae:.2f}, Accuracy(\u00b15yrs) {acc_cs5:.1f}%)")

print("Training complete. Best val MAE:", best_val_mae)

Starting training: 7694 batches per epoch, 8 epochs total
  epoch 1 batch 0/7694  loss=0.1284  elapsed=0s
  epoch 1 batch 20/7694  loss=0.0886  elapsed=2s
  epoch 1 batch 40/7694  loss=0.1110  elapsed=3s
  epoch 1 batch 60/7694  loss=0.0738  elapsed=4s
  epoch 1 batch 80/7694  loss=0.0980  elapsed=5s
  epoch 1 batch 100/7694  loss=0.0798  elapsed=6s
  epoch 1 batch 120/7694  loss=0.0523  elapsed=7s
  epoch 1 batch 140/7694  loss=0.0812  elapsed=8s
  epoch 1 batch 160/7694  loss=0.1020  elapsed=10s
  epoch 1 batch 180/7694  loss=0.0767  elapsed=11s
  epoch 1 batch 200/7694  loss=0.0781  elapsed=12s
  epoch 1 batch 220/7694  loss=0.0792  elapsed=13s
  epoch 1 batch 240/7694  loss=0.0936  elapsed=14s
  epoch 1 batch 260/7694  loss=0.0765  elapsed=15s
  epoch 1 batch 280/7694  loss=0.0453  elapsed=16s
  epoch 1 batch 300/7694  loss=0.0861  elapsed=18s
  epoch 1 batch 320/7694  loss=0.0740  elapsed=19s
  epoch 1 batch 340/7694  loss=0.0834  elapsed=20s
  epoch 1 batch 360/7694  loss=0.0787 

In [3]:
# Test the fine-tuned model on a new image, right here in Colab
from google.colab import files
import cv2
from PIL import Image
import torchvision.transforms as T
import torch

print("Upload a test image")
uploaded = files.upload()
test_img_path = list(uploaded.keys())[0]

# Detect face using the same detector
img_bgr = cv2.imread(test_img_path)
result = detector.predict(img_bgr)
face_indices = [ind for ind in range(len(result.yolo_results.boxes))
                 if result.yolo_results.names[int(result.yolo_results.boxes[ind].cls)] == "face"]

assert face_indices, "No face detected in this image, try another one."

best_ind = max(face_indices, key=lambda ind: result.get_bbox_by_ind(ind, *img_bgr.shape[:2])[2:].prod())
x1, y1, x2, y2 = result.get_bbox_by_ind(best_ind, *img_bgr.shape[:2])
h, w = img_bgr.shape[:2]
bw, bh = x2 - x1, y2 - y1
MARGIN = 0.15
x1 = max(0, int(x1 - bw * MARGIN)); y1 = max(0, int(y1 - bh * MARGIN))
x2 = min(w, int(x2 + bw * MARGIN)); y2 = min(h, int(y2 + bh * MARGIN))
face_crop = img_bgr[y1:y2, x1:x2]

# Preprocess and predict
tf = T.Compose([T.Resize((input_size, input_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
face_pil = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
input_tensor = tf(face_pil).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    out = model(input_tensor)
    if out.dim() > 1:
        out = out.squeeze(-1)
    pred_age = out.item() * (max_age - min_age) + avg_age

print(f"Predicted age: {pred_age:.1f} years")

# Show the detected face crop
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
plt.title(f"Predicted age: {pred_age:.1f}")
plt.axis("off")
plt.show()

Upload a test image


Saving veena.jpg to veena.jpg


NameError: name 'detector' is not defined

In [28]:
# Cell 11: Download the fine-tuned checkpoint to your computer
from google.colab import files
files.download("/content/checkpoints/mivolo_finetuned_best.pth.tar")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>